# 14 — Unit Testing, Fixtures, and Mocking

Tests turn assumptions into executable checks. Python's built-in unittest framework provides assertions, setup hooks, and mocks. External pytest can be introduced later without making the notebooks depend on it.

*Intermediate · 75–120 min*

## You will learn

- Structure unittest.TestCase assertions
- Test success, invalid input, and boundaries
- Use setUp and temporary resources
- Mock external boundaries without calling the network

## Examples

### 1. Writing a first unit test

A test method begins with test_ and calls assertions such as assertEqual and assertRaises. In Jupyter, use a TestLoader and TextTestRunner rather than unittest.main, because the notebook kernel supplies its own command-line arguments.

In [ ]:
import unittest

def normalize_name(text):
    if not text.strip():
        raise ValueError("empty name")
    return text.strip().title()

class NameTests(unittest.TestCase):
    def test_normalization(self):
        self.assertEqual(normalize_name("  ada "), "Ada")
    def test_reject_empty(self):
        with self.assertRaises(ValueError):
            normalize_name("  ")

suite = unittest.defaultTestLoader.loadTestsFromTestCase(NameTests)
result = unittest.TextTestRunner(verbosity=1).run(suite)
assert result.wasSuccessful()

**Think about it:** What does assertRaises test that assertEqual cannot?

### 2. Setup and state isolation

Each test should operate on fresh state to avoid execution-order dependence. setUp runs before each test method; tearDown can clean resources. Prefer tiny fixtures and temporary directories for files.

In [ ]:
import unittest

class Basket:
    def __init__(self):
        self.items = []
    def add(self, price):
        self.items.append(price)
    def total(self):
        return sum(self.items)

class BasketTests(unittest.TestCase):
    def setUp(self):
        self.basket = Basket()
    def test_empty(self):
        self.assertEqual(self.basket.total(), 0)
    def test_two_items(self):
        self.basket.add(3)
        self.basket.add(4)
        self.assertEqual(self.basket.total(), 7)

assert unittest.TextTestRunner(verbosity=0).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(BasketTests)
).wasSuccessful()

**Think about it:** Why would sharing one Basket across all tests introduce flaky results?

### 3. Mocking a dependency

A mock substitutes a collaborator that would otherwise perform I/O or nondeterministic work. Test observable behavior and important interactions without over-specifying implementation details. MagicMock can count calls and set return values.

In [ ]:
from unittest.mock import Mock

def send_report(sender, text):
    if not text:
        raise ValueError("report is empty")
    return sender.send(text)

fake_sender = Mock()
fake_sender.send.return_value = "queued"
assert send_report(fake_sender, "Results ready") == "queued"
fake_sender.send.assert_called_once_with("Results ready")
print(fake_sender.send.call_count)

**Think about it:** Which part of this code would be replaced by a real email or HTTP service?

### 4. Subtests and test matrices

subTest lets a single test method check many inputs while reporting which case failed. Choose meaningful boundary data rather than only ordinary examples. Avoid tests that depend on the current date or network.

In [ ]:
import unittest

def clamp(value, low, high):
    if low > high:
        raise ValueError("invalid range")
    return max(low, min(value, high))

class ClampTests(unittest.TestCase):
    def test_cases(self):
        for value, expected in [(-10, 0), (5, 5), (15, 10)]:
            with self.subTest(value=value):
                self.assertEqual(clamp(value, 0, 10), expected)

assert unittest.TextTestRunner(verbosity=0).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(ClampTests)
).wasSuccessful()

**Think about it:** Why is this more informative than testing only the value 5?

## Exercises

**Exercise 1.** Test a function that validates percentage discounts.

In [ ]:
# Exercise 1
# Implement your solution here, and add tests.

**Exercise 2.** Write a unittest.TestCase for a small shopping cart.

In [ ]:
# Exercise 2
# Implement your solution here, and add tests.

**Exercise 3.** Mock a notification sender and assert it was called with the correct message.

In [ ]:
# Exercise 3
# Implement your solution here, and add tests.

**Exercise 4.** Write a table of edge cases and turn it into subTest checks.

In [ ]:
# Exercise 4
# Implement your solution here, and add tests.

## Common mistakes

- Use unittest's loader and runner inside notebooks; unittest.main reads notebook arguments.
- Do not depend on real network services in unit tests.
- Tests should be deterministic and isolated from each other.

**Next:** [15 — SQLite, SQL, and Transactions](15_sqlite_and_transactions.ipynb)